# 3 · Experiments

From the two processed datasets to the numbers the paper reports. Six experiments,
seven model families, five seeds each.

The two processed datasets are **released**, so this notebook runs without the
PitchBook extraction: `data/processed/dataset_window.csv` and
`dataset_nowindow.csv` are all it needs. The same runs happen from the command
line:

```bash
python scripts/pipeline.py experiments --setting window
python scripts/pipeline.py experiments --setting all
python scripts/pipeline.py experiments --setting window --wandb   # log to W&B
```

## Weights & Biases is optional

With `USE_WANDB = False` the runs happen locally and the metrics are printed and
kept in memory, which is enough for every table below. With it on, a sweep agent
drives the runs and everything is logged to your entity and project, read from
`.env`. Nothing else changes: it is the same function either way.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.integrate
import yaml

from src.experiments import SETTINGS, load_setting
from src.training import make_train, run_grid
from src.utils import (
    clear_split_cache,
    compare_metrics,
    compute_wilcoxon_table,
    get_split,
    plot_correlation_heatmap,
    plot_shap_comparison,
    summarize_metrics,
)

# numpy 2 removed np.trapz; some of the plotting dependencies still call it.
if not hasattr(np, "trapz"):
    np.trapz = scipy.integrate.trapezoid

USE_WANDB = False

CONFIG = yaml.safe_load(open("config/config.yaml"))
FREQUENCY = CONFIG["frequency_encoding"]

# Everything get_split needs, in one place: the split geometry and the
# frequency-encoding settings. The encoding is fitted per seed on the training
# split alone, so the threshold below is the only knob that decides which
# categories survive and which are pooled into "Others".
split_kwargs = {
    "test_size": CONFIG["test_size"],
    "cache_dir": "tmp/splits",
    "categorical_columns": FREQUENCY["columns"],
    "min_frequency": FREQUENCY["min_frequency"],
    "other_label": FREQUENCY["other_label"],
}

# The two stores accumulate across the experiments run in this session, and the
# comparison cells at the bottom read them. They live here so that they survive a
# module reload.
metrics_store: dict = {}
shap_store: dict = {}

for name, description in SETTINGS.items():
    print(f"{name:15} {description}")

---
## Choosing an experiment

One line, and it decides everything below. Two of the six are the datasets as they
come out of the preprocessing, two are ablations of the bias-controlled one, and
two are the controls that take the 2x2 apart by swapping the target between the two
datasets on `CompanyID` — legitimate because both carry the same firms and the same
columns.

Run the cells below once per experiment: the stores keep what each one produced.

In [ ]:
SETTING = "window"

dataset = load_setting(SETTING, CONFIG)
X = dataset.drop(["CompanyID", "Target"], axis=1)
y = dataset["Target"]
print(f"{SETTING}: {SETTINGS[SETTING]}")
print(f"{len(dataset):,} firms, {X.shape[1]} features, base rate {y.mean():.3f}")

The base rate travels with the definition of the target, and no model here tunes
its decision threshold, so F1, precision and recall move with it on their own.
**AUC is the metric to read across the label axis.**

In [ ]:
corr_matrix = plot_correlation_heatmap(dataset)

---
## Splits, imputation and scaling

One split per evaluation seed: 60% train, 20% validation for the threshold, 20%
test. The frequency encoding, the KNN imputer and the scaler are each fitted on
that seed's training rows alone, inside `get_split`, so no held-out row contributes
to the transform later applied to it.

The imputation costs minutes per split and every model revisits the same split, so
`get_split` caches under `tmp/splits`. The cache file name carries a fingerprint of
the columns and of the encoding settings, so editing a threshold in `config.yaml`
or rebuilding the datasets invalidates it on its own; `REBUILD_SPLITS` wipes this
experiment's splits anyway.

In [ ]:
REBUILD_SPLITS = False

if REBUILD_SPLITS:
    removed = clear_split_cache(cache_dir=split_kwargs["cache_dir"], tag=SETTING)
    print(f"cache cleared for {SETTING!r}: {removed} file(s) removed")

for seed in CONFIG["seeds"]:
    get_split(X, y, seed, tag=SETTING, **split_kwargs)
    print(f"seed {seed}: split ready")

---
## Training

Seven families crossed with five seeds: 35 runs. Each run draws its own seed, and
that seed drives **both** the split and the model's randomness, so the five runs of
a model are five independent replications rather than five reruns of one partition
— which is what lets the mean ± std tell a real gap between two models from
split-to-split noise.

SHAP is computed on the first evaluation seed only. Those tables answer a different
question from the mean ± std ones: which features move when the window is removed,
not how much the metrics vary across splits.

In [ ]:
if USE_WANDB:
    import os

    import wandb
    from dotenv import load_dotenv

    load_dotenv()
    sweep_id = wandb.sweep(
        CONFIG["sweep_settings"], entity=os.getenv("entity"), project=os.getenv("project")
    )
    train = make_train(SETTING, X, y, CONFIG, split_kwargs, metrics_store, shap_store)
    wandb.agent(sweep_id, function=train, count=35)
else:
    run_grid(SETTING, X, y, CONFIG, split_kwargs, metrics_store, shap_store)

---
## The results of this experiment

One row per model, every cell the `mean ± std` across the evaluation seeds. The
standard deviation is the sample one, so a `± 0.000` cell means a single run rather
than a model insensitive to the split, and the `Seeds` column tells the two apart.

In [ ]:
results = summarize_metrics(metrics_store, SETTING)
print(f"experiment {SETTING!r} — mean ± std across seeds {CONFIG['seeds']}")
print(results.to_string(index=False))

---
## Comparing the experiments

Run the experiments you want to compare first — go back to the *Choosing an
experiment* cell, change `SETTING`, and run down to here again. The stores keep
what each one produced.

`window` is (aligned features, aligned label) and `nowindow` is (leaked, leaked),
so comparing those two measures the two leaks **summed**. The two controls hold one
axis fixed at a time: against `leaklabel` only the definition of the target
changes, against `leakfeat` only the features do, at a constant base rate.

In [ ]:
for other, label in (
    ("nowindow", "both leaks"),
    ("leaklabel", "leaked label"),
    ("leakfeat", "leaked features"),
):
    if any(tag == other for _, tag in metrics_store):
        print(f"\n── window vs {other} ({label})")
        print(compare_metrics(metrics_store, "window", other, label_b=label).to_string(index=False))

The two ablations answer a different question: not how much bias a leak adds, but
how much a family of features is worth.

In [ ]:
for other in ("noteam", "nocompetitors"):
    if any(tag == other for _, tag in metrics_store):
        print(f"\n── window vs {other}")
        print(compare_metrics(metrics_store, "window", other).to_string(index=False))

---
## What the models look at

The SHAP comparison shows which features carry the weight in each experiment, and
the Wilcoxon table tests whether the difference between two experiments is
significant. The test pairs rows **within** one explained sample, which is why SHAP
is computed on a single seed: pooling five would change what the test measures.

In [ ]:
fig = plot_shap_comparison(shap_store)
plt.show()

In [ ]:
print(compute_wilcoxon_table(shap_store).to_string(index=False))

---
## Where the numbers come from

- the panel: **`1_panel_construction.ipynb`**, or `python scripts/pipeline.py panel --both`;
- the datasets: **`2_dataset_construction.ipynb`**, or `python scripts/pipeline.py datasets`;
- everything at once: `python scripts/pipeline.py all`.